# Stage C — Fine-tune Criterion Cross-Encoder

Fine-tunes BiomedBERT-base on `criteria_r1_labels.jsonl` (R1-Distill-7B annotations)
to predict 5-class criterion eligibility labels for (patient, criterion) pairs.

**Evaluation target:** NDCG@10 on TREC22 (clf-v4 eval_baseline gate = 0.6388)

**Pipeline at inference:**
1. clf-v4 retrieves top-50 trials per topic
2. Criterion scorer predicts P(label) for each (patient, criterion) pair
3. Per-trial score = Σ P(label)·weight aggregated over all criteria
4. Optionally: clf-v4 score + criterion adjustment (combined signal)

**Label score weights:** included=+1, not_excluded=+1, not_included=−1, excluded=−2, NEI=0

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers datasets scikit-learn accelerate tqdm
!pip install -q sympy==1.13.1

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # HuggingFace token (READ sufficient for BiomedBERT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT        = '/content/drive/MyDrive/ct_data23'
LABELS_PATH      = f'{DATA_ROOT}/criteria_r1_labels.jsonl'
CKPT_DIR         = f'{DATA_ROOT}/criterion_clf_v1'   # save best checkpoint here

BASE_MODEL       = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
CLF_CHECKPOINT   = 'semaj83/ctmatch-clf-v4'          # for NDCG eval

MAX_LENGTH       = 512
BATCH_SIZE       = 32
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 5
WARMUP_RATIO     = 0.1
VAL_TOPIC_FRAC   = 0.2   # fraction of topics held out for val-NDCG
TOP_K_EVAL       = 50    # trials per topic for NDCG eval
COMBINE_CLF      = True  # also eval clf-v4-score + criterion adjustment

# Label order, weights, and class indices
LABELS     = ['included', 'not_included', 'excluded', 'not_excluded', 'not_enough_information']
LABEL2ID   = {l: i for i, l in enumerate(LABELS)}
ID2LABEL   = {i: l for i, l in enumerate(LABELS)}
# Weights for expected-score aggregation
LABEL_W    = {'included': 1.0, 'not_included': -1.0, 'excluded': -2.0,
              'not_excluded': 1.0, 'not_enough_information': 0.0}
SCORE_VEC  = [LABEL_W[l] for l in LABELS]   # aligned with model output order

## Pre-flight: annotation quality check

The labels file was written across multiple runs with two different parse versions.
Check for systematic problems before committing a training run.

In [ ]:
import json
from collections import Counter

recs = []
with open(LABELS_PATH) as f:
    for line in f:
        recs.append(json.loads(line))

total = len(recs)
n_no_reasoning = sum(1 for r in recs if not r.get('reasoning'))
n_truncated    = sum(1 for r in recs if r.get('truncated'))
label_dist     = Counter(r['label'] for r in recs)
src_dist       = Counter(r['source'] for r in recs)

print(f'Total records      : {total:,}')
print(f'Unique topics      : {len({r["topic_id"] for r in recs})}')
print(f'Source split       : {dict(src_dist)}')
print(f'No reasoning       : {n_no_reasoning} ({100*n_no_reasoning/total:.1f}%)  ← should be <10%')
print(f'Truncated          : {n_truncated} ({100*n_truncated/total:.1f}%)')
print()
print('Label distribution:')
for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:5,}  ({100*count/total:.1f}%)')

nei_pct = 100 * label_dist.get('not_enough_information', 0) / total
exc_pct = 100 * label_dist.get('excluded', 0) / total
print(f'\nNEI rate: {nei_pct:.1f}%  (Claude simple-label baseline: 41.8%)')
print(f'excluded: {exc_pct:.1f}%  — will use class weighting to compensate')

# Check if no-reasoning records are clustered by source or topic (sign of parse bug)
no_reas_topics = Counter(r['topic_id'] for r in recs if not r.get('reasoning'))
if no_reas_topics:
    print(f'\nNo-reasoning records spread across {len(no_reas_topics)} topics'
          f' (top 5: {no_reas_topics.most_common(5)})')
    no_reas_src = Counter(r['source'] for r in recs if not r.get('reasoning'))
    print(f'No-reasoning by source: {dict(no_reas_src)}')

In [ ]:
# Filter records from the buggy-parse run.
#
# All 770 empty-reasoning records fall in 8 trec21 topics that were annotated
# before the parse fix. The checkpoint skipped them on re-run, leaving stale data.
# Their labels are also suspect: the old code ran parse_label() on the full R1
# output (reasoning + </think> + label), so reasoning text could have shadowed the
# real label. Dropping these topics is the safe call; 10,480 clean records remain.

bad_topics = {r['topic_id'] for r in recs if not r.get('reasoning')}
print(f'Topics with empty reasoning (buggy-parse run): {sorted(bad_topics)}')

recs_clean = [r for r in recs if r['topic_id'] not in bad_topics]
dropped    = len(recs) - len(recs_clean)
print(f'Dropped {dropped} records ({100*dropped/len(recs):.1f}%)  →  {len(recs_clean):,} clean records remain')

# Rebuild label_dist from clean records (used for class weights in cell-train)
label_dist = Counter(r['label'] for r in recs_clean)
print('\nClean label distribution:')
for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:5,}  ({100*count/len(recs_clean):.1f}%)')

recs = recs_clean  # downstream cells use recs

## Train / val split by topic
Split on topic_id (not record) to prevent patient-text leakage across splits.

In [ ]:
import random

random.seed(42)
all_topics = sorted({r['topic_id'] for r in recs})
n_val      = max(1, int(len(all_topics) * VAL_TOPIC_FRAC))
val_topics = set(random.sample(all_topics, n_val))
trn_topics = set(all_topics) - val_topics

trn_recs = [r for r in recs if r['topic_id'] in trn_topics]
val_recs = [r for r in recs if r['topic_id'] in val_topics]

print(f'Train: {len(trn_topics)} topics, {len(trn_recs):,} records')
print(f'Val  : {len(val_topics)} topics, {len(val_recs):,} records')

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

class CriterionDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r   = self.records[idx]
        enc = tokenizer(
            r['topic_text'] if 'topic_text' in r else '',
            f"{r['crit_type'].upper()}: {r['criterion']}",
            # longest_first, not only_first: the naive parser occasionally emits an
            # entire eligibility block as one "criterion" longer than max_length,
            # which only_first cannot truncate under the limit (tokenizer raises)
            truncation='longest_first',
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(LABEL2ID[r['label']], dtype=torch.long),
        }

# Records don't carry topic_text inline — need to join from qrels
topic2text = {}
QRELS_PATH = f'{DATA_ROOT}/unified_qrels.jsonl'
with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        topic2text[rec['topic_id']] = rec['topic_text']

for r in recs:
    r['topic_text'] = topic2text.get(r['topic_id'], '')

trn_ds = CriterionDataset(trn_recs)
val_ds = CriterionDataset(val_recs)
print(f'Train dataset: {len(trn_ds):,}  |  Val dataset: {len(val_ds):,}')

## Training

`excluded` is only 2.5% of labels but carries the highest penalty weight (−2).
Use inverse-frequency class weighting so the model doesn't collapse it.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Class weights: inverse frequency, normalised
label_counts = np.array([label_dist.get(l, 1) for l in LABELS], dtype=float)
class_weights = torch.tensor(1.0 / label_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum() * len(LABELS)
class_weights = class_weights.to(device)
print('Class weights:', {l: f'{w:.2f}' for l, w in zip(LABELS, class_weights.cpu().tolist())})

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps  = len(trn_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

best_val_loss = float('inf')
os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    # ── train ──
    model.train()
    trn_loss = 0.0
    for batch in tqdm(trn_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} train'):
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(input_ids=batch['input_ids'],
                       attention_mask=batch['attention_mask']).logits
        loss = loss_fn(logits, batch['labels'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        trn_loss += loss.item()

    # ── val ──
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} val'):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(input_ids=batch['input_ids'],
                           attention_mask=batch['attention_mask']).logits
            val_loss += loss_fn(logits, batch['labels']).item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_labels.extend(batch['labels'].cpu().tolist())

    avg_trn = trn_loss / len(trn_loader)
    avg_val = val_loss / len(val_loader)
    print(f'\nEpoch {epoch+1}  trn_loss={avg_trn:.4f}  val_loss={avg_val:.4f}')
    print(classification_report(
        all_labels, all_preds,
        target_names=LABELS, zero_division=0,
    ))

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
        print(f'  ✓ saved best checkpoint (val_loss={avg_val:.4f})')

## NDCG@10 eval on val topics

Loads the best checkpoint and scores the val topics using criterion aggregation.
Compares against clf-v4 alone and (optionally) clf-v4 + criterion adjustment.

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset

# Load best criterion scorer
crit_tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
crit_model     = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(device)
crit_model.eval()

# Load clf-v4 for baseline scores
from transformers import AutoModelForSequenceClassification as AMSC
clf_tokenizer = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf_model     = AMSC.from_pretrained(CLF_CHECKPOINT).to(device)
clf_model.eval()
clf_relevant  = [int(k) for k, v in clf_model.config.id2label.items() if v == 'relevant'][0]

# Load corpus
index2docid_ds = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
doc_texts_ds   = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',   split='train')
index2docid    = [row['text'].strip() for row in index2docid_ds]
docid2text     = {nct_id: doc_texts_ds[idx]['text'] for idx, nct_id in enumerate(index2docid)}

# Load criteria
nctid2crit = {}
with open(f'{DATA_ROOT}/criteria_data.jsonl') as f:
    for line in f:
        r = json.loads(line)
        nctid2crit[r['nct_id']] = r

# Load qrels for val topics  (doc_id field = NCT ID)
val_topic2rel = {}
with open(QRELS_PATH) as f:
    for line in f:
        r = json.loads(line)
        if r['topic_id'] not in val_topics:
            continue
        val_topic2rel.setdefault(r['topic_id'], {})[r['doc_id']] = r['label']

score_vec = torch.tensor(SCORE_VEC, device=device)

def clf_scores(topic_text, nct_ids, batch_size=64):
    scores = []
    texts  = [docid2text.get(n, '') for n in nct_ids]
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = clf_tokenizer(
            [topic_text] * len(batch), batch,
            padding=True, truncation=True, max_length=512,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            logits = clf_model(**enc).logits
        scores.extend(F.softmax(logits, -1)[:, clf_relevant].cpu().tolist())
    return scores

def criterion_score(topic_text, nct_id, batch_size=64):
    crit  = nctid2crit.get(nct_id, {'include_criteria': [], 'exclude_criteria': []})
    crits = [(f"INCLUSION: {c}") for c in crit['include_criteria']] + \
            [(f"EXCLUSION: {c}") for c in crit['exclude_criteria']]
    if not crits:
        return 0.0
    crit_scores = []
    for i in range(0, len(crits), batch_size):
        batch = crits[i:i+batch_size]
        enc = crit_tokenizer(
            [topic_text] * len(batch), batch,
            # longest_first — matches training; only_first raises on the rare
            # criterion longer than max_length
            truncation='longest_first', max_length=MAX_LENGTH,
            padding=True, return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            probs = F.softmax(crit_model(**enc).logits, -1)  # (n, 5)
        crit_scores.extend((probs * score_vec).sum(-1).cpu().tolist())
    return float(np.mean(crit_scores))

In [ ]:
from sklearn.metrics import ndcg_score

# Cache per-topic arrays so the fusion-weight sweep below reuses them for free
topic_scores = {}   # tid -> (true_rel, clf_s, crit_s)

ndcg_clf, ndcg_crit, ndcg_combined = [], [], []

def znorm(x):
    s = x.std()
    return (x - x.mean()) / s if s > 0 else x - x.mean()

for tid in tqdm(sorted(val_topics), desc='Val NDCG'):
    topic_text = topic2text[tid]
    rel        = val_topic2rel.get(tid, {})
    nct_ids    = [n for n in rel if n in docid2text]
    if not nct_ids:
        continue
    true_rel = np.array([rel[n] for n in nct_ids])

    clf_s  = np.array(clf_scores(topic_text, nct_ids))
    crit_s = np.array([criterion_score(topic_text, n) for n in nct_ids])
    topic_scores[tid] = (true_rel, clf_s, crit_s)

    ndcg_clf.append(ndcg_score([true_rel], [clf_s],  k=10))
    ndcg_crit.append(ndcg_score([true_rel], [crit_s], k=10))
    if COMBINE_CLF:
        combined = znorm(clf_s) + znorm(crit_s)
        ndcg_combined.append(ndcg_score([true_rel], [combined], k=10))

print(f'\nVal topics: {len(ndcg_clf)}')
print(f'NDCG@10  clf-v4 alone   : {np.mean(ndcg_clf):.4f}   (inflated: clf-v4 trained on these topics)')
print(f'NDCG@10  criterion alone: {np.mean(ndcg_crit):.4f}')
if COMBINE_CLF:
    print(f'NDCG@10  combined (α=1) : {np.mean(ndcg_combined):.4f}')
print(f'\nGate (clf-v4 eval_baseline TREC22): 0.6388')
print('Note: val NDCG uses judged-pool docs only — run eval_baseline for authoritative TREC22 number')

In [ ]:
# ── Fusion-weight sweep: combined(α) = znorm(clf) + α·znorm(crit) ────────────
#
# Answers whether the criterion signal carries ANY complementary information:
#   best α ≈ 0        → current scorer adds nothing over clf-v4 (pure noise)
#   best α > 0 with a
#   real NDCG bump    → complementary signal exists; a learned aggregator has
#                       something to amplify
# Uses the arrays cached by the cell above — no model calls, runs in seconds.

alphas = np.round(np.arange(0.0, 2.01, 0.05), 2)
sweep  = []
for a in alphas:
    vals = [ndcg_score([tr], [znorm(cs) + a * znorm(crs)], k=10)
            for tr, cs, crs in topic_scores.values()]
    sweep.append(np.mean(vals))
sweep = np.array(sweep)

best_i = int(sweep.argmax())
print(f'α=0.00 (clf-v4 only) : {sweep[0]:.4f}')
print(f'α=1.00 (equal)       : {sweep[np.where(alphas == 1.0)[0][0]]:.4f}')
print(f'best α={alphas[best_i]:.2f}         : {sweep[best_i]:.4f}  '
      f'(Δ vs clf-v4: {sweep[best_i] - sweep[0]:+.4f})')

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.5))
plt.plot(alphas, sweep, marker='.', ms=4)
plt.axhline(sweep[0], color='gray', ls='--', lw=1, label='clf-v4 alone')
plt.axvline(alphas[best_i], color='red', ls=':', lw=1, label=f'best α={alphas[best_i]:.2f}')
plt.xlabel('α (criterion weight)'); plt.ylabel('mean NDCG@10 (val)')
plt.legend(); plt.tight_layout(); plt.show()

# Per-topic check at best α: is the gain broad or driven by 1-2 topics?
if best_i > 0:
    a = alphas[best_i]
    deltas = [(tid, ndcg_score([tr], [znorm(cs) + a * znorm(crs)], k=10)
                    - ndcg_score([tr], [cs], k=10))
              for tid, (tr, cs, crs) in topic_scores.items()]
    n_up = sum(1 for _, d in deltas if d > 1e-6)
    n_dn = sum(1 for _, d in deltas if d < -1e-6)
    print(f'Topics improved: {n_up}  |  hurt: {n_dn}  |  unchanged: {len(deltas)-n_up-n_dn}')
    for tid, d in sorted(deltas, key=lambda x: -abs(x[1]))[:5]:
        print(f'  {tid:15s} Δ={d:+.4f}')